In [1]:
import pandas as pd
import s3fs

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("🏆 Generando tablas adicionales de alto impacto para la Capa Gold...")

# Cargar la tabla maestra desde Silver en memoria
df_trans = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_transaction_transformed.parquet", storage_options=storage_opts)
df_id = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_identity_transformed.parquet", storage_options=storage_opts)
df_master = pd.merge(df_trans, df_id, on="TransactionID", how="left")

# ==============================================================================
# TABLA GOLD 3: Impacto Económico del Fraude (por Marca de Tarjeta y Rango de Ticket)
# ==============================================================================
gold_impacto_economico = df_master.groupby(['card4', 'rango_ticket'], as_index=False).agg(
    total_transacciones=('TransactionID', 'count'),
    transacciones_fraudulentas=('isFraud', 'sum'),
    monto_total_expuesto_pen=('monto_pen', 'sum'),
    monto_perdido_fraude_pen=('monto_pen', lambda x: x[df_master.loc[x.index, 'isFraud'] == 1].sum())
)

gold_impacto_economico['tasa_fraude_pct'] = (
    gold_impacto_economico['transacciones_fraudulentas'] / gold_impacto_economico['total_transacciones'] * 100
).round(2)

gold_impacto_economico['pct_monto_perdido'] = (
    gold_impacto_economico['monto_perdido_fraude_pen'] / gold_impacto_economico['monto_total_expuesto_pen'] * 100
).round(2)

# Guardar en Capa Gold MinIO
gold_impacto_economico.to_parquet(
    f"s3://{BUCKET_NAME}/gold/kpis_impacto_economico.parquet", 
    index=False, 
    storage_options=storage_opts
)


# ==============================================================================
# TABLA GOLD 4: Transacciones por Minuto / Hora (Velocidad Operativa y Pico de Carga)
# ==============================================================================
# Agregamos por día de la semana y hora exacta para medir la velocidad transaccional
gold_velocidad = df_master.groupby(['dia_semana', 'hora_dia'], as_index=False).agg(
    total_transacciones=('TransactionID', 'count'),
    fraudes_detectados=('isFraud', 'sum'),
    volumen_dinero_pen=('monto_pen', 'sum')
)

# Promedio de transacciones por hora (asumiendo los días registrados)
# Como cada hora tiene 60 minutos, calculamos el estimado de transacciones por minuto en cada franja horaria
gold_velocidad['transacciones_por_minuto_promedio'] = (gold_velocidad['total_transacciones'] / 60.0).round(2)

# Guardar en Capa Gold MinIO
gold_velocidad.to_parquet(
    f"s3://{BUCKET_NAME}/gold/kpis_velocidad_transaccional.parquet", 
    index=False, 
    storage_options=storage_opts
)

print("✨ ¡Tablas económicas y de velocidad añadidas a la Capa Gold con éxito!")

print("\n📊 Vista previa - Impacto Económico del Fraude (S/):")
display(gold_impacto_economico.sort_values(by='monto_perdido_fraude_pen', ascending=False).head(6))

print("\n📊 Vista previa - Velocidad Operativa (Transacciones por Minuto/Hora):")
display(gold_velocidad.sort_values(by='total_transacciones', ascending=False).head(6))

🏆 Generando tablas adicionales de alto impacto para la Capa Gold...
✨ ¡Tablas económicas y de velocidad añadidas a la Capa Gold con éxito!

📊 Vista previa - Impacto Económico del Fraude (S/):


,card4,rango_ticket,total_transacciones,transacciones_fraudulentas,monto_total_expuesto_pen,monto_perdido_fraude_pen,tasa_fraude_pct,pct_monto_perdido
19,visa,4_Alto (>$200),60026,2609,104549304.0,4.685926e+06,4.35,4.48
18,visa,3_Medio ($50-$200),192344,5731,70768280.0,2.248623e+06,2.98,3.18
15,mastercard,4_Alto (>$200),29434,1175,52350856.0,1.894216e+06,3.99,3.62
14,mastercard,3_Medio ($50-$200),91146,2645,33156884.0,1.032838e+06,2.90,3.12
11,discover,4_Alto (>$200),2273,276,5223179.0,6.026833e+05,12.14,11.54
17,visa,2_Bajo ($15-$50),121257,4140,16369055.0,5.212522e+05,3.41,3.18



📊 Vista previa - Velocidad Operativa (Transacciones por Minuto/Hora):


,dia_semana,hora_dia,total_transacciones,fraudes_detectados,volumen_dinero_pen,transacciones_por_minuto_promedio
18,Friday,18,7339,236,4631956.5,122.32
20,Friday,20,7287,223,4329721.0,121.45
17,Friday,17,7283,273,4425842.0,121.38
19,Friday,19,7265,242,4223637.5,121.08
21,Friday,21,7241,228,4227375.5,120.68
16,Friday,16,7028,213,4130586.0,117.13
